# Fundamentals of Data Science  
## Week 6: Web Scraping

---

## Table of Contents
1. Learning Objectives
2. Context & Motivation
3. Conceptual Foundation
4. Demonstration
5. Hands-on
6. Reflection
7. Deliverable Checklist
---

# 1) Learning Objectives

เมื่อจบคาบนี้ นักศึกษาจะสามารถ

1. อธิบายได้ว่า Web Scraping คือการดึงข้อมูลจาก **โครงสร้าง DOM** ไม่ใช่การ copy หน้าเว็บ
2. ใช้เครื่องมือ Python เพื่อดึง “ข้อมูลข่าวเชิงเวลา” มาเป็น external feature ได้
3. ออกแบบ feature จากข่าว (news volume / event signal) โดยอธิบายเหตุผลและข้อจำกัดได้
4. ปฏิบัติการดึงข้อมูลจากเว็บอย่างมีจริยธรรมและมีขอบเขตที่เหมาะสม

# 2) Context & Motivation

จนถึงตอนนี้ เรามี:

* PM2.5 time series ระยะยาว (2001–2024)
* Feature เชิงเวลาและเชิงโดเมนจากข้อมูลตัวเลข

**ปัญหาที่ข้อมูลยังตอบไม่ได้**

* ทำไมบางช่วง PM2.5 สูง “พร้อมกันทั้งเมือง”
* ทำไมบางช่วงตัวเลขสูง แต่สังคมกลับไม่สนใจ
* ทำไมบางช่วงสื่อพูดถึงมากผิดปกติ

ข้อมูลตัวเลขเพียงอย่างเดียว **ไม่บอกบริบททางสังคม**

> Web Scraping ในคาบนี้  
> ไม่ได้มีเป้าหมายเพื่อ “เก็บข่าวให้ครบ”  
> แต่เพื่อสร้าง **ตัวแทน (proxy)** ของเหตุการณ์และความสนใจของสังคม

---

# 3) Conceptual Foundation

## 3.1 Web Scraping คืออะไร

**Web Scraping = การเลือกข้อมูลจากโครงสร้างของเว็บ (DOM)**
ไม่ใช่:

* การ copy หน้าเว็บ
* การโหลดไฟล์ข้อมูลที่เตรียมไว้ให้

สิ่งที่เราทำจริง ๆ คือ:

> “เลือก element บางส่วนจากหน้าเว็บ แล้วแปลงเป็นข้อมูล”

---

## 3.2 DOM (Document Object Model): แก่นของ Web Scraping

![Image](https://www.w3schools.com/js/pic_htmltree.gif)

**DOM คืออะไร**

* DOM คือโครงสร้างแบบต้นไม้ (tree) ที่แทนหน้าเว็บ 1 หน้า
* ทุกหน้าเว็บ = DOM 1 ต้น

ตัวอย่างโครงสร้างในเชิงแนวคิด

```
html
 └── body
      ├── h1        ← หัวข้อข่าว
      ├── div
      │    ├── p    ← เนื้อข่าว
      │    └── p
      └── footer
```

> Web Scraping = การ “เดินในต้นไม้” แล้วหยิบโหนดที่ต้องการ

---

## 3.3 Tag, Attribute, Text (3 คำที่ต้องเข้าใจ)

| แนวคิด    | ความหมาย    | ตัวอย่าง         |
| --------- | ----------- | ---------------- |
| Tag       | โหนดใน DOM  | `h1`, `p`, `div` |
| Attribute | ตัวระบุโหนด | `class="title"`  |
| Text      | ข้อมูลจริง  | “ค่าฝุ่นวิกฤต”   |

---

## 3.4 เชื่อม DOM กับโค้ดที่ใช้จริง

```python
soup.find("h1")
```

หมายถึง

> เลือกโหนด `h1` ตัวแรกใน DOM

```python
soup.find_all("p")
```

หมายถึง

> เลือกทุกโหนด `p`

```python
soup.select("div.news p")
```

หมายถึง

> เลือก `p` ที่อยู่ใต้ `div` class=`news` เท่านั้น

**ประเด็นสำคัญ**

* ถ้า DOM เปลี่ยน → selector เดิมอาจใช้ไม่ได้
* นี่คือเหตุผลที่ Web Scraping “พังได้เสมอ”

---

## 3.5 Ethics & Legal Considerations

Web Scraping **ไม่ใช่เรื่องเทคนิคอย่างเดียว**

หลักที่ควรใช้ในการทำงาน

* ยึดหลักการใช้ข้อมูลสาธารณะ
* ดึงด้วยความถี่ต่ำ (rate limit)
* เก็บเฉพาะ metadata (title, date, url) และส่วนที่ต้องการ
* อ้างอิงแหล่งข่าวทุกครั้ง

> เป้าหมายคือ “วิเคราะห์เชิงวิชาการ” ไม่ใช่ทำ data harvesting

---

# 4) Demonstration

**ขั้นตอน**

1. เปิดหน้าเว็บ
2. Inspect DOM ด้วย DevTools
3. หาสิ่งที่ต้องการ

   * หัวข้อข่าวอยู่ตรงไหน
   * วันที่อยู่ตรงไหน
4. เลือก element ด้วย BeautifulSoup

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://news.google.com/rss/search?q=PM2.5+Bangkok+site:thairath.co.th+after:2021-01-01+before:2025-01-01&hl=th&gl=TH&ceid=TH:th"
xml = requests.get(url).text
soup = BeautifulSoup(xml, "xml")
items = soup.select("item")
len(items)

100

In [5]:
rows = []

for it in items:
  rows.append({
    "title": it.title.text if it.title else None,
    "link": it.link.text if it.link else None,
    "published": it.pubDate.text if it.pubDate else None,
    "source": it.source.text if it.source else None
  })

import pandas as pd
df_news = pd.DataFrame(rows)
df_news.head()

,title,link,published,source
0,ขุ่นพระ! กรุงเทพฯ 7 ปี คนกรุงสูดพิษ PM2.5 ไปถึ...,https://news.google.com/rss/articles/CBMiWEFVX...,"Thu, 14 Nov 2024 08:00:00 GMT",Thairath
1,เผยที่มา สาเหตุค่าฝุ่น PM 2.5 ใน กทม. สูงเกินม...,https://news.google.com/rss/articles/CBMiaEFVX...,"Mon, 05 Feb 2024 08:00:00 GMT",Thairath
2,เตือนฝุ่น PM 2.5 ถล่มกรุงเทพฯ-เชียงใหม่ เอลนีโ...,https://news.google.com/rss/articles/CBMiXkFVX...,"Sun, 29 Oct 2023 07:00:00 GMT",Thairath
3,ฝุ่นพิษ PM 2.5 ถล่มหนัก กรุงเทพฯ-ปริมณฑล 20 พื...,https://news.google.com/rss/articles/CBMiY0FVX...,"Sat, 16 Jan 2021 08:00:00 GMT",Thairath
4,ฝุ่น PM 2.5 เช้าวันนี้ กทม. มีแนวโน้มเพิ่มขึ้น...,https://news.google.com/rss/articles/CBMiY0FVX...,"Wed, 05 Apr 2023 07:00:00 GMT",Thairath


**สิ่งที่ต้องสังเกต**

* เราไม่ได้รู้โครงสร้างเว็บล่วงหน้า
* ต้อง “ดู DOM ก่อนเขียนโค้ด”

---

# 5) Hands-on

## Task 1: สร้าง Feature เชิงเวลา

* จำนวนข่าวต่อวัน
* จำนวนข่าวต่อเดือน
* `high_media_attention` (1 เมื่อข่าวมากผิดปกติ)

In [6]:
df_news["published"] = pd.to_datetime(df_news["published"])

news_per_day = (
    df_news.groupby(df_news["published"].dt.date)
    .size()
    .reset_index(name="news_count_day")
)

news_per_month = (
    df_news.groupby(df_news["published"].dt.to_period("M"))
    .size()
    .reset_index(name="news_count_month")
)

s = news_per_day["news_count_day"]
news_per_day["high_media_attention"] = (s > s.mean() + s.std()).astype(int)

news_per_day.head()

,published,news_count_day,high_media_attention
0,2021-01-16,3,1
1,2021-01-25,1,0
2,2021-02-02,1,0
3,2021-02-13,1,0
4,2021-03-09,1,0


## Task 2: บันทึกข้อมูลที่ได้เป็น `PM2.5_news.csv`

In [7]:
news_per_day.to_csv("PM2.5_news.csv", index=False)

---

# 6) Reflection

ตอบคำถามต่อไปนี้ใน Notebook

1. Feature จาก “จำนวนข่าว” สะท้อนอะไรที่ PM2.5 ตัวเลขไม่บอก
2. ช่วงใดที่ข่าวมาก แต่ PM2.5 ไม่สูง (หรือกลับกัน)
3. ข้อมูลข่าวมี bias แบบใดบ้าง และส่งผลอย่างไรต่อการตีความ

---

1. ความสนใจของสังคมต่อผลกระทบทางสุขภาพ นโยบาย หรือเหตุการณ์พิเศษที่ค่าฝุ่นแบบตัวเลขไม่สามารถอธิบายได้
2. ช่วงต้นปี 2021 และต้นปี 2022 ค่าฝุ่นเยอะแต่ข่าวเกี่ยวกับฝุ่น PM2.5 น้อย อาจเป็นเพราะช่วงเวลานั้นมีข่าวใหญ่เรื่องอื่นอยู่
3. Selection Bias จากการใช้ข่าวบางแหล่ง/บางพื้นที่ ทำให้ไม่ครอบคลุมสถานการณ์จริง / Measurement Bias เพราะใช้จำนวนข่าวแทนค่าฝุ่นจริง ขึ้นกับการนำเสนอของสื่อ / Missing Data Bias บางวันไม่มีข่าวแม้ฝุ่นสูง

# 7) Deliverable Checklist

* [ ] Notebook ที่ทำ Hands-on และ Reflection แล้ว
* [ ] ไฟล์ `PM2.5_news.csv`

---